# Extreme events analysis 1: Tropical nights

### Imports:
Libraries required to run the script

In [18]:
import sys
sys.path.append("..")
from utils.Urban_Rural_functions import get_daily_min_or_max_fields
import numpy as np
import xarray as xr
from urclimask.utils import kelvin2degC

### Variables:

In [19]:
#City name and data info:
variable = 'tas'
driving_model = 'ERA5'
scenario = 'evaluation'
member = 'r1i1p1f1'
institution = 'BCCR-UCAN'
model = 'WRF451R-CI4C'
version = 'v1-r1'
time_period = '1hr'
data_version = 'v20240710'
city='Paris'

# Obtaination of the domain from the city_name
if city == 'Paris' or city_name == 'Barcelona' or city_name == 'Prage':
    # domain = 'ALPX-3i' 
    domain = 'ALPX-3f'#For my data, the current version is called ALPX-3i.
elif city == 'Bergen' or city_name == 'Hamburg' or city_name == 'NewCastle':
    domain = 'NSEA-3i-'

# Tropical night threshold
T_tn = 20 #Temperature in Celsius


### Read data

In [20]:
# Open data:
path = 'data/'
file = variable + '_' + domain + '_' + driving_model + '_' + scenario + '_' + member + '_' + institution + '_' + model + '_' + version + '_' + time_period + '_' + city + '_cropped_interpolated.nc'
ds = xr.open_dataset(path+file)
#Convert Kelvin degrees to celsius:
ds = kelvin2degC(ds, variable)#K->ºC
data=ds[variable]


# # Minimum and maximum temperarture obtention:
ds_min = get_daily_min_or_max_fields(data, 'minimum')
ds_max = get_daily_min_or_max_fields(data, 'maximum')

In [21]:
## Apply the mask:

# Read the mask:
output_dir = '../utils/urclimask_Results'# Name of the output masks directory
mask = xr.open_dataset(f"{output_dir}/{city}_{domain}_{model}_urban_rural_regridded_mask.nc")

#Definition of urban and rural masks:
rural_mask = mask.urmask == 0
urban_mask = mask.urmask == 1

# Apply it:
ds_min_rural = ds_min.where(rural_mask)
ds_min_urban = ds_min.where(urban_mask)
ds_max_rural = ds_max.where(rural_mask)
ds_max_urban = ds_max.where(urban_mask)


# Only summer:

In [22]:
## Extract seasons:
season_months = [6, 7, 8]
season_ds_min_rural = ds_min_rural.sel(time=ds_min_rural.time.dt.month.isin(season_months))
season_ds_min_urban = ds_min_urban.sel(time=ds_min_urban.time.dt.month.isin(season_months))
# season_ds_max_rural = ds_min_rural.sel(time=ds_max_rural.time.dt.month.isin(season_months))
# season_ds_max_urban = ds_min_urban.sel(time=ds_max_urban.time.dt.month.isin(season_months))
season_ds_min = ds_min.sel(time=ds_min.time.dt.month.isin(season_months))
# season_ds_max = ds_max.sel(time=ds_max.time.dt.month.isin(season_months))

In [23]:
#Promediate spacially:
season_ds_min_rural = season_ds_min_rural.mean(dim="lon").mean(dim="lat")
season_ds_min_urban = season_ds_min_urban.mean(dim="lon").mean(dim="lat")
# season_ds_max_rural = season_ds_max_rural.mean(dim="lon").mean(dim="lat")
# season_ds_max_urban = season_ds_max_urban.mean(dim="lon").mean(dim="lat")
season_ds_min = season_ds_min.mean(dim="lon").mean(dim="lat")
# season_ds_max = season_ds_max.mean(dim="lon").mean(dim="lat")

In [24]:
# all_variables = ["season_ds_min_rural", "season_ds_min_urban", "season_ds_max_rural", "season_ds_max_urban", "season_ds_min", "season_ds_max"]
active_variables = ["season_ds_min_rural", "season_ds_min_urban", "season_ds_min"]

In [25]:
for variable_name in active_variables:
    variable = globals()[variable_name]

    # Diagnóstico: confirma que cada variable es distinta
    # print(variable_name, id(variable), float(variable.mean().compute()))

    n_total = variable.sizes["time"]          # más seguro que len(variable)
    n_tropical = int((variable > T_tn).sum(dim="time").compute().item())
    percentage = 100 * n_tropical / n_total if n_total > 0 else np.nan
    print(f"{variable_name}: {n_tropical} noches tropicales de {n_total} noches totales ({percentage:.2f} %)")


season_ds_min_rural: 133 noches tropicales de 2300 noches totales (5.78 %)
season_ds_min_urban: 149 noches tropicales de 2300 noches totales (6.48 %)
season_ds_min: 110 noches tropicales de 2300 noches totales (4.78 %)


# All seasons:

In [26]:
## Extract seasons:Spring
season_months = [12, 1, 2]
spring_ds_min_rural = ds_min_rural.sel(time=ds_min_rural.time.dt.month.isin(season_months))
spring_ds_min_urban = ds_min_urban.sel(time=ds_min_urban.time.dt.month.isin(season_months))
spring_ds_max_rural = ds_min_rural.sel(time=ds_max_rural.time.dt.month.isin(season_months))
spring_ds_max_urban = ds_min_urban.sel(time=ds_max_urban.time.dt.month.isin(season_months))
spring_ds_min = ds_min.sel(time=ds_min.time.dt.month.isin(season_months))
spring_ds_max = ds_max.sel(time=ds_max.time.dt.month.isin(season_months))

## Extract seasons:Summer
season_months = [6, 7, 8]
summer_ds_min_rural = ds_min_rural.sel(time=ds_min_rural.time.dt.month.isin(season_months))
summer_ds_min_urban = ds_min_urban.sel(time=ds_min_urban.time.dt.month.isin(season_months))
summer_ds_max_rural = ds_min_rural.sel(time=ds_max_rural.time.dt.month.isin(season_months))
summer_ds_max_urban = ds_min_urban.sel(time=ds_max_urban.time.dt.month.isin(season_months))
summer_ds_min = ds_min.sel(time=ds_min.time.dt.month.isin(season_months))
summer_ds_max = ds_max.sel(time=ds_max.time.dt.month.isin(season_months))

## Extract seasons:Autumn
season_months = [9, 10, 11]
autumn_ds_min_rural = ds_min_rural.sel(time=ds_min_rural.time.dt.month.isin(season_months))
autumn_ds_min_urban = ds_min_urban.sel(time=ds_min_urban.time.dt.month.isin(season_months))
autumn_ds_max_rural = ds_min_rural.sel(time=ds_max_rural.time.dt.month.isin(season_months))
autumn_ds_max_urban = ds_min_urban.sel(time=ds_max_urban.time.dt.month.isin(season_months))
autumn_ds_min = ds_min.sel(time=ds_min.time.dt.month.isin(season_months))
autumn_ds_max = ds_max.sel(time=ds_max.time.dt.month.isin(season_months))


## Extract seasons:Winter
season_months = [12, 1, 2]
winter_ds_min_rural = ds_min_rural.sel(time=ds_min_rural.time.dt.month.isin(season_months))
winter_ds_min_urban = ds_min_urban.sel(time=ds_min_urban.time.dt.month.isin(season_months))
winter_ds_max_rural = ds_min_rural.sel(time=ds_max_rural.time.dt.month.isin(season_months))
winter_ds_max_urban = ds_min_urban.sel(time=ds_max_urban.time.dt.month.isin(season_months))
winter_ds_min = ds_min.sel(time=ds_min.time.dt.month.isin(season_months))
winter_ds_max = ds_max.sel(time=ds_max.time.dt.month.isin(season_months))

In [27]:
# Average spatially — spring:
spring_ds_min_rural = spring_ds_min_rural.mean(dim="lon").mean(dim="lat")
spring_ds_min_urban = spring_ds_min_urban.mean(dim="lon").mean(dim="lat")
spring_ds_max_rural = spring_ds_max_rural.mean(dim="lon").mean(dim="lat")
spring_ds_max_urban = spring_ds_max_urban.mean(dim="lon").mean(dim="lat")
spring_ds_min = spring_ds_min.mean(dim="lon").mean(dim="lat")
spring_ds_max = spring_ds_max.mean(dim="lon").mean(dim="lat")

# Average spatially — summer:
summer_ds_min_rural = summer_ds_min_rural.mean(dim="lon").mean(dim="lat")
summer_ds_min_urban = summer_ds_min_urban.mean(dim="lon").mean(dim="lat")
summer_ds_max_rural = summer_ds_max_rural.mean(dim="lon").mean(dim="lat")
summer_ds_max_urban = summer_ds_max_urban.mean(dim="lon").mean(dim="lat")
summer_ds_min = summer_ds_min.mean(dim="lon").mean(dim="lat")
summer_ds_max = summer_ds_max.mean(dim="lon").mean(dim="lat")

# Average spatially — autumn:
autumn_ds_min_rural = autumn_ds_min_rural.mean(dim="lon").mean(dim="lat")
autumn_ds_min_urban = autumn_ds_min_urban.mean(dim="lon").mean(dim="lat")
autumn_ds_max_rural = autumn_ds_max_rural.mean(dim="lon").mean(dim="lat")
autumn_ds_max_urban = autumn_ds_max_urban.mean(dim="lon").mean(dim="lat")
autumn_ds_min = autumn_ds_min.mean(dim="lon").mean(dim="lat")
autumn_ds_max = autumn_ds_max.mean(dim="lon").mean(dim="lat")

# Average spatially — winter:
winter_ds_min_rural = winter_ds_min_rural.mean(dim="lon").mean(dim="lat")
winter_ds_min_urban = winter_ds_min_urban.mean(dim="lon").mean(dim="lat")
winter_ds_max_rural = winter_ds_max_rural.mean(dim="lon").mean(dim="lat")
winter_ds_max_urban = winter_ds_max_urban.mean(dim="lon").mean(dim="lat")
winter_ds_min = winter_ds_min.mean(dim="lon").mean(dim="lat")
winter_ds_max = winter_ds_max.mean(dim="lon").mean(dim="lat")

In [28]:
all_variables = [
    "spring_ds_min_rural", "spring_ds_min_urban", "spring_ds_max_rural", "spring_ds_max_urban", "spring_ds_min", "spring_ds_max",
    "summer_ds_min_rural", "summer_ds_min_urban", "summer_ds_max_rural", "summer_ds_max_urban", "summer_ds_min", "summer_ds_max",
    "autumn_ds_min_rural", "autumn_ds_min_urban", "autumn_ds_max_rural", "autumn_ds_max_urban", "autumn_ds_min", "autumn_ds_max",
    "winter_ds_min_rural", "winter_ds_min_urban", "winter_ds_max_rural", "winter_ds_max_urban", "winter_ds_min", "winter_ds_max"
]

active_variables = [
    "spring_ds_min_rural", "spring_ds_min_urban", "spring_ds_min",
    "summer_ds_min_rural", "summer_ds_min_urban", "summer_ds_min",
    "autumn_ds_min_rural", "autumn_ds_min_urban", "autumn_ds_min",
    "winter_ds_min_rural", "winter_ds_min_urban", "winter_ds_min"
]

In [29]:
for variable_name in active_variables:
    variable = globals()[variable_name]

    # Diagnóstico: confirma que cada variable es distinta
    # print(variable_name, id(variable), float(variable.mean().compute()))

    n_total = variable.sizes["time"]          # más seguro que len(variable)
    n_tropical = int((variable > T_tn).sum(dim="time").compute().item())
    percentage = 100 * n_tropical / n_total if n_total > 0 else np.nan
    print(f"{variable_name}: {n_tropical} noches tropicales de {n_total} noches totales ({percentage:.2f} %)")

spring_ds_min_rural: 0 noches tropicales de 2256 noches totales (0.00 %)
spring_ds_min_urban: 0 noches tropicales de 2256 noches totales (0.00 %)
spring_ds_min: 0 noches tropicales de 2256 noches totales (0.00 %)
summer_ds_min_rural: 133 noches tropicales de 2300 noches totales (5.78 %)
summer_ds_min_urban: 149 noches tropicales de 2300 noches totales (6.48 %)
summer_ds_min: 110 noches tropicales de 2300 noches totales (4.78 %)
autumn_ds_min_rural: 17 noches tropicales de 2275 noches totales (0.75 %)
autumn_ds_min_urban: 17 noches tropicales de 2275 noches totales (0.75 %)
autumn_ds_min: 15 noches tropicales de 2275 noches totales (0.66 %)
winter_ds_min_rural: 0 noches tropicales de 2256 noches totales (0.00 %)
winter_ds_min_urban: 0 noches tropicales de 2256 noches totales (0.00 %)
winter_ds_min: 0 noches tropicales de 2256 noches totales (0.00 %)


In [30]:
# Extremos al 95%:

from scipy.stats import genextreme

MIN_VALID_POINTS = 20  # minimum number of valid data points required to attempt the GEV fit

for variable_name in all_variables:
    variable = globals()[variable_name]

    # Bring Dask-backed data into memory (if applicable) and flatten to 1D
    data_values = variable.compute().values.ravel()

    # Remove NaN values before fitting
    data_values = data_values[~np.isnan(data_values)]

    n_valid = data_values.size

    # Skip the fit if there is not enough valid data
    if n_valid < MIN_VALID_POINTS:
        print(f"{variable_name}: datos insuficientes para el ajuste GEV "
              f"({n_valid} valores válidos, se requieren al menos {MIN_VALID_POINTS})")
        continue

    # Fit the GEV distribution using maximum likelihood estimation
    shape, location, scale = genextreme.fit(data_values)

    # Compute the value corresponding to the 95th percentile of the fitted GEV
    percentile_95 = genextreme.ppf(0.95, shape, loc=location, scale=scale)

    print(f"{variable_name}: {n_valid} datos válidos utilizados")
    print(f"  Parámetros GEV -> shape (c): {shape:.4f}, loc: {location:.4f}, scale: {scale:.4f}")
    print(f"  Valor en el percentil 95 (GEV ajustada): {percentile_95:.4f}")
    print()

spring_ds_min_rural: 1232 datos válidos utilizados
  Parámetros GEV -> shape (c): 0.2782, loc: -0.1392, scale: 4.0454
  Valor en el percentil 95 (GEV ajustada): 8.0376

spring_ds_min_urban: 1232 datos válidos utilizados
  Parámetros GEV -> shape (c): 0.2660, loc: 0.4637, scale: 3.8827
  Valor en el percentil 95 (GEV ajustada): 8.4359

spring_ds_max_rural: 1232 datos válidos utilizados
  Parámetros GEV -> shape (c): 0.2782, loc: -0.1392, scale: 4.0454
  Valor en el percentil 95 (GEV ajustada): 8.0376

spring_ds_max_urban: 1232 datos válidos utilizados
  Parámetros GEV -> shape (c): 0.2660, loc: 0.4637, scale: 3.8827
  Valor en el percentil 95 (GEV ajustada): 8.4359

spring_ds_min: 1232 datos válidos utilizados
  Parámetros GEV -> shape (c): 0.2819, loc: -0.2808, scale: 4.0603
  Valor en el percentil 95 (GEV ajustada): 7.8879

spring_ds_max: 1232 datos válidos utilizados
  Parámetros GEV -> shape (c): 2.6053, loc: 15.9857, scale: 3.3099
  Valor en el percentil 95 (GEV ajustada): 17.2556
